 Gated Linear Unit (GLU) — Convolutional Text Classification

In [23]:
# CELL 1 — Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [24]:
%pip install --upgrade datasets pyarrow numpy

Note: you may need to restart the kernel to use updated packages.


In [25]:
# CELL 2 — Load dataset (AG News — topic classification: World/Sports/Business/Sci-Tech)
from datasets import load_dataset

# Use the direct community-verified structural path
ds = load_dataset("wangrongsheng/ag_news")

train_data = ds["train"]
test_data = ds["test"]

class_names = ["World", "Sports", "Business", "Sci/Tech"]
print(f"Train samples: {len(train_data)} | Test samples: {len(test_data)}")
print("Example:", train_data[0]["text"][:150], "| Label:", class_names[train_data[0]["label"]])

Train samples: 120000 | Test samples: 7600
Example: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again. | Label: Business


In [26]:
# CELL 3 — Build vocabulary and encode text
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9' ]", " ", text)
    return text.split()

MAX_VOCAB_SIZE = 15000
counter = Counter()
subset_for_vocab = train_data.shuffle(seed=42).select(range(10000))
for item in subset_for_vocab:
    counter.update(tokenize(item["text"]))

most_common = counter.most_common(MAX_VOCAB_SIZE - 2)
vocab = {"<pad>": 0, "<unk>": 1}
for word, _ in most_common:
    vocab[word] = len(vocab)

MAX_LEN = 60

def encode(text, vocab, max_len=MAX_LEN):
    tokens = tokenize(text)
    ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokens[:max_len]]
    if len(ids) < max_len:
        ids += [vocab["<pad>"]] * (max_len - len(ids))
    return ids

print(f"Vocab size: {len(vocab)}")

Vocab size: 15000


In [27]:
# CELL 4 — Dataset and DataLoaders
class AGNewsDataset(Dataset):
    def __init__(self, hf_data, vocab, max_len=MAX_LEN):
        self.data = hf_data
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        ids = encode(item["text"], self.vocab, self.max_len)
        return torch.tensor(ids, dtype=torch.long), torch.tensor(item["label"], dtype=torch.long)

train_subset = train_data.shuffle(seed=42).select(range(20000))
test_subset = test_data.shuffle(seed=42).select(range(3000))

train_dataset = AGNewsDataset(train_subset, vocab)
test_dataset = AGNewsDataset(test_subset, vocab)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f"Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")

Train samples: 20000 | Test samples: 3000


In [28]:
# CELL 5 — Model definition
class GLU1D(nn.Module):
    """Gated Linear Unit: splits conv output in half, one half gates the other via sigmoid.
    This lets the network learn to selectively pass or block information — the same
    gating idea behind LSTM/GRU gates, but applied convolutionally and in parallel
    rather than sequentially. Popularized by Gated ConvNets (Dauphin et al.) for language modeling."""
    def __init__(self, in_channels, out_channels, kernel_size, padding):
        super().__init__()
        # Output double the channels: one half becomes the "value", other half the "gate"
        self.conv = nn.Conv1d(in_channels, out_channels * 2, kernel_size, padding=padding)

    def forward(self, x):
        conv_out = self.conv(x)                          # [batch, out_channels*2, seq_len]
        value, gate = conv_out.chunk(2, dim=1)             # split down the channel dimension
        return value * torch.sigmoid(gate)                 # GLU: value gated by sigmoid(gate)

class GLUTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_classes=4, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.glu_block1 = GLU1D(embed_dim, 128, kernel_size=3, padding=1)
        self.glu_block2 = GLU1D(128, 128, kernel_size=3, padding=1)
        self.glu_block3 = GLU1D(128, 128, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveMaxPool1d(1)  # global max pooling over sequence
        self.fc = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        embedded = self.embedding(x)              # [batch, seq_len, embed_dim]
        x = embedded.permute(0, 2, 1)              # [batch, embed_dim, seq_len] — conv1d expects channels-first
        x = self.glu_block1(x)                     # gated conv block 1
        x = self.glu_block2(x)                      # gated conv block 2
        x = self.glu_block3(x)                       # gated conv block 3
        x = self.pool(x).squeeze(-1)                  # [batch, 128]
        x = self.dropout(x)
        return self.fc(x)                              # raw logits

model = GLUTextClassifier(vocab_size=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(model)

GLUTextClassifier(
  (embedding): Embedding(15000, 128, padding_idx=0)
  (glu_block1): GLU1D(
    (conv): Conv1d(128, 256, kernel_size=(3,), stride=(1,), padding=(1,))
  )
  (glu_block2): GLU1D(
    (conv): Conv1d(128, 256, kernel_size=(3,), stride=(1,), padding=(1,))
  )
  (glu_block3): GLU1D(
    (conv): Conv1d(128, 256, kernel_size=(3,), stride=(1,), padding=(1,))
  )
  (pool): AdaptiveMaxPool1d(output_size=1)
  (fc): Linear(in_features=128, out_features=4, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


In [29]:
# CELL 6 — Training loop
epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for sequences, labels in train_loader:
        sequences, labels = sequences.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(sequences)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} — Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}%")

Epoch 1/5 — Loss: 0.8705 | Train Acc: 63.62%
Epoch 2/5 — Loss: 0.4677 | Train Acc: 83.19%
Epoch 3/5 — Loss: 0.3207 | Train Acc: 88.94%
Epoch 4/5 — Loss: 0.2171 | Train Acc: 92.79%
Epoch 5/5 — Loss: 0.1426 | Train Acc: 95.28%


In [30]:
# CELL 7 — Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for sequences, labels in test_loader:
        sequences, labels = sequences.to(device), labels.to(device)
        logits = model(sequences)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 84.97%


In [31]:
# CELL 8 — Try it on custom headlines
def predict_topic(text, model, vocab):
    model.eval()
    ids = encode(text, vocab)
    tensor = torch.tensor([ids], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
    return class_names[pred], probs[0][pred].item()

sample_headlines = [
    "The central bank raised interest rates by half a point today.",
    "The striker scored a hat-trick in the final minutes of the match.",
    "Scientists discover a new exoplanet orbiting a distant star.",
    "Tensions rise as diplomats meet for emergency peace talks."
]

for headline in sample_headlines:
    topic, conf = predict_topic(headline, model, vocab)
    print(f"Headline: {headline}\nPredicted topic: {topic} (confidence: {conf:.2f})\n")

Headline: The central bank raised interest rates by half a point today.
Predicted topic: Business (confidence: 0.89)

Headline: The striker scored a hat-trick in the final minutes of the match.
Predicted topic: Sports (confidence: 1.00)

Headline: Scientists discover a new exoplanet orbiting a distant star.
Predicted topic: Sci/Tech (confidence: 1.00)

Headline: Tensions rise as diplomats meet for emergency peace talks.
Predicted topic: World (confidence: 0.99)

